In [ ]:
# DEBERTA-V3-BASE



# CELL 1 INSTALL
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "scikit-learn"])

pkgs = [
    "transformers>=4.48.0",
    "accelerate>=1.2.0",
    "evaluate",
    "datasets>=3.1.0",
    "sentence-transformers",
    "bertopic",
    "umap-learn",
    "hdbscan==0.8.41",
    "iterative-stratification",
    "rapidfuzz",
    "scikit-learn>=1.8.0",
    "sentencepiece",             # required by DebertaV2Tokenizer
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# CELL 2 LOAD DATA
import pickle, numpy as np, os, time
from datetime import timedelta

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark   = True

DATA = "/kaggle/input/datasets/mabhishek0/nlp-genre-project-data/"
SAVE = "/kaggle/working/"

with open(f"{DATA}data_splits.pkl", "rb") as f:
    splits = pickle.load(f)

X_train, y_train = splits["X_train"], splits["y_train"]
X_val,   y_val   = splits["X_val"],   splits["y_val"]
X_test,  y_test  = splits["X_test"],  splits["y_test"]

with open(f"{DATA}mlb.pkl", "rb") as f:
    mlb = pickle.load(f)

TARGET_GENRES = list(mlb.classes_)
NUM_LABELS    = len(TARGET_GENRES)  # 12
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Labels: {NUM_LABELS} | Device: {device}")


# CELL 3 UTILITIES
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DebertaV2Tokenizer, DebertaV2Model,
    get_cosine_schedule_with_warmup
)
from sklearn.metrics import f1_score, hamming_loss, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")


def fmt(seconds):
    return str(timedelta(seconds=int(seconds)))


def eta(elapsed_steps, total_steps, elapsed_seconds):
    if elapsed_steps == 0:
        return "calculating..."
    return fmt((elapsed_seconds / elapsed_steps) * (total_steps - elapsed_steps))


def get_pos_weight(y_train, device):
    counts = y_train.sum(axis=0)
    weight = (len(y_train) - counts) / (counts + 1e-8)
    return torch.FloatTensor(weight).to(device)


def tune_threshold_per_genre(probs, labels, num_labels, target_genres):
    """
    Per-genre threshold tuning — sweeps 0.20 → 0.85 independently per genre.
    Proven to drive large F1 gains: drove DistilBERT from 0.6459 → 0.6831
    on identical model weights purely from this change.
    Returns array of shape (num_labels,).
    """
    best_thresholds = np.zeros(num_labels)
    print("  Per-genre threshold tuning:")
    for i in range(num_labels):
        best_f1, best_t = 0.0, 0.5
        for t in np.arange(0.20, 0.85, 0.05):
            preds = (probs[:, i] > t).astype(int)
            f1    = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds[i] = best_t
        print(f"    {target_genres[i]:<20} best_t={best_t:.2f}  F1={best_f1:.4f}")
    return best_thresholds


def evaluate(model, loader, thresholds, model_name, device):
    """Full test-set evaluation using per-genre thresholds."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            # Plain FP32 forward — no autocast for DeBERTa on T4
            logits = model(batch["input_ids"], batch["attention_mask"])
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs > thresholds).astype(int)
            all_preds.append(preds)
            all_labels.append(batch["labels"].cpu().numpy())
    P, L = np.vstack(all_preds), np.vstack(all_labels)
    print(f"\n=== {model_name} Test Results (per-genre thresholds) ===")
    print(f"Micro-F1:     {f1_score(L, P, average='micro', zero_division=0):.4f}")
    print(f"Macro-F1:     {f1_score(L, P, average='macro', zero_division=0):.4f}")
    print(f"Hamming Loss: {hamming_loss(L, P):.4f}")
    print(classification_report(L, P, target_names=TARGET_GENRES, zero_division=0))


# CELL 4 — VRAM-CACHED DATASET + MODEL
DB_MODEL_NAME = "microsoft/deberta-v3-base"
DB_MAX_LEN    = 256


class UltraFastDebertaDataset(Dataset):
    """
    Tokenizes all texts once, caches input_ids + attention_mask + labels
    directly in GPU VRAM. __getitem__ is a pure VRAM tensor index —
    zero CPU work and zero host-device transfer per batch.

    tolist() converts numpy array slices → list[str] required by
    DebertaV2Tokenizer (verified failure without this).
    """
    def __init__(self, texts, labels, tokenizer, max_len=256, device="cuda"):
        self.labels = torch.FloatTensor(labels).to(device)
        print(f"Tokenizing {len(texts)} texts → VRAM...")
        # Fix: tolist() required — DebertaV2Tokenizer rejects numpy slices
        text_list = texts.tolist() if hasattr(texts, "tolist") else list(texts)
        encodings = tokenizer(
            text_list,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        self.input_ids      = encodings["input_ids"].to(device)
        self.attention_mask = encodings["attention_mask"].to(device)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx]
        }


class DeBERTaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Fix A: force FP32 at load time — prevents internal FP16 residuals
        # from DeBERTa's disentangled attention layers
        self.db_encoder = DebertaV2Model.from_pretrained(
            DB_MODEL_NAME,
            torch_dtype=torch.float32
        )
        # Fix F: gradient checkpointing — recomputes activations during
        # backward. Required to fit FP32 DeBERTa at batch_size=32 in 15.8GB.
        self.db_encoder.gradient_checkpointing_enable()

        self.db_drop = nn.Dropout(0.1)
        self.db_clf  = nn.Linear(self.db_encoder.config.hidden_size, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out = self.db_encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Fix B: .float() defensive cast — DeBERTa disentangled attention
        # emits FP16 tensors internally even without autocast active.
        # Without this: RuntimeError: mat1 and mat2 must have same dtype
        cls = out.last_hidden_state[:, 0, :].float()
        return self.db_clf(self.db_drop(cls))


# CELL 5 — LOAD INTO VRAM
db_tokenizer = DebertaV2Tokenizer.from_pretrained(DB_MODEL_NAME)

tok_start = time.time()
print("\n--- Loading data directly into GPU VRAM ---")
train_ds = UltraFastDebertaDataset(X_train, y_train, db_tokenizer, max_len=DB_MAX_LEN, device=device)
val_ds   = UltraFastDebertaDataset(X_val,   y_val,   db_tokenizer, max_len=DB_MAX_LEN, device=device)
test_ds  = UltraFastDebertaDataset(X_test,  y_test,  db_tokenizer, max_len=DB_MAX_LEN, device=device)
print(f"All splits loaded in {fmt(time.time() - tok_start)}")

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM after data load: {used:.1f} GB / {total:.1f} GB ({100*used/total:.1f}% used)")

# num_workers=0 required on Kaggle — silent deadlock with any value > 0
# pin_memory=False — tensors already in VRAM
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


# CELL 6 — TRAINING
db_model = DeBERTaClassifier().to(device)
# Fix C: ensure ALL parameters are FP32 after device transfer
db_model = db_model.float()

# Print VRAM after model load
used_after_model = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after model load: {used_after_model:.1f} GB / {total:.1f} GB ({100*used_after_model/total:.1f}% used)")

db_criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(y_train, device))

# Differential LRs — encoder backbone low (preserve pretrained weights),
# classifier head high (trained from scratch)
db_optimizer = AdamW([
    {"params": db_model.db_encoder.parameters(), "lr": 1e-5, "weight_decay": 0.01},
    {"params": db_model.db_drop.parameters(),    "lr": 1e-4, "weight_decay": 0.0},
    {"params": db_model.db_clf.parameters(),     "lr": 1e-4, "weight_decay": 0.0},
])

DB_EPOCHS      = 8
db_total_steps = len(train_loader) * DB_EPOCHS

# Cosine scheduler — smoother LR decay than linear
db_scheduler = get_cosine_schedule_with_warmup(
    db_optimizer,
    num_warmup_steps=int(0.1 * db_total_steps),
    num_training_steps=db_total_steps
)

db_best_macro            = 0.0
db_best_thresholds_saved = np.full(NUM_LABELS, 0.5)
db_patience              = 5
db_no_improve            = 0
train_start              = time.time()
global_step              = 0

print("=" * 65)
print(f"DeBERTa-v3-base optimal training started.")
print(f"Total steps: {db_total_steps} | Steps/epoch: {len(train_loader)}")
print(f"Batch size: 32 | Max len: {DB_MAX_LEN} | Precision: FP32")
print(f"Epochs: {DB_EPOCHS} | Patience: {db_patience} | Scheduler: Cosine")
print(f"Gradient checkpointing: ON (required for FP32 DeBERTa on T4)")
print(f"No autocast, no GradScaler (DeBERTa-v3 FP32 constraint)")
print(f"Threshold: per-genre sweep 0.20 → 0.85")
print(f"Estimated total time: ~3-5 hours")
print("=" * 65)

for epoch in range(DB_EPOCHS):
    epoch_start  = time.time()
    running_loss = 0.0
    db_model.train()

    print(f"\n{'─'*65}")
    print(f"EPOCH {epoch+1}/{DB_EPOCHS}  |  Elapsed: {fmt(time.time() - train_start)}")
    print(f"{'─'*65}")

    for step, batch in enumerate(train_loader):
        step_start = time.time()
        db_optimizer.zero_grad()

        # Plain FP32 forward — NO autocast wrapper (Fix D)
        # Tensors already in VRAM — no .to(device) needed
        db_logits = db_model(batch["input_ids"], batch["attention_mask"])
        db_loss   = db_criterion(db_logits, batch["labels"])

        # Plain backward — NO GradScaler (Fix D)
        db_loss.backward()
        torch.nn.utils.clip_grad_norm_(db_model.parameters(), max_norm=1.0)
        db_optimizer.step()
        db_scheduler.step()

        running_loss += db_loss.item()
        global_step  += 1
        step_time     = time.time() - step_start

        if (step + 1) % 100 == 0:
            avg_loss       = running_loss / (step + 1)
            epoch_elapsed  = time.time() - epoch_start
            total_elapsed  = time.time() - train_start
            steps_left_ep  = len(train_loader) - (step + 1)
            epoch_eta_secs = (epoch_elapsed / (step + 1)) * steps_left_ep

            print(
                f"  Step {step+1:>4}/{len(train_loader)}"
                f"  |  Loss: {avg_loss:.4f}"
                f"  |  Step: {step_time:.2f}s"
                f"  |  Epoch ETA: {fmt(epoch_eta_secs)}"
                f"  |  Total ETA: {eta(global_step, db_total_steps, total_elapsed)}"
            )

    epoch_duration = time.time() - epoch_start
    print(f"\nEpoch {epoch+1} training done in {fmt(epoch_duration)}")

    # ── Validation with per-genre threshold tuning ────────────────────────────
    val_start = time.time()
    db_model.eval()
    all_logits, all_labels_val = [], []

    with torch.no_grad():
        for b in val_loader:
            # Plain FP32 inference — no autocast (Fix D)
            db_vlogits = db_model(b["input_ids"], b["attention_mask"])
            all_logits.append(db_vlogits.cpu().numpy())
            all_labels_val.append(b["labels"].cpu().numpy())

    val_duration  = time.time() - val_start
    all_logits_np = np.vstack(all_logits)
    all_labels_np = np.vstack(all_labels_val)
    all_probs_np  = 1 / (1 + np.exp(-all_logits_np))

    # Per-genre threshold tuning
    best_t_epoch = tune_threshold_per_genre(
        all_probs_np, all_labels_np, NUM_LABELS, TARGET_GENRES
    )

    best_preds_epoch = (all_probs_np > best_t_epoch).astype(int)
    ep_macro_f1      = f1_score(
        all_labels_np, best_preds_epoch, average="macro", zero_division=0
    )

    total_elapsed = time.time() - train_start
    print(f"\nValidation done in {fmt(val_duration)}")
    print(f"Epoch {epoch+1}/{DB_EPOCHS}  |  Val Macro-F1: {ep_macro_f1:.4f}")
    print(f"Total elapsed: {fmt(total_elapsed)}  |  Remaining ETA: {eta(global_step, db_total_steps, total_elapsed)}")

    if ep_macro_f1 > db_best_macro:
        db_best_macro            = ep_macro_f1
        db_best_thresholds_saved = best_t_epoch.copy()
        db_no_improve            = 0
        torch.save(db_model.state_dict(), f"{SAVE}deberta_v3_optimal_best.pt")
        print("  -> Saved best model")
        print(f"  -> Thresholds: {dict(zip(TARGET_GENRES, best_t_epoch.round(2)))}")
    else:
        db_no_improve += 1
        print(f"  -> No improvement ({db_no_improve}/{db_patience})")

    if db_no_improve >= db_patience:
        print("Early stopping triggered.")
        break

total_time = time.time() - train_start
print(f"\n{'='*65}")
print(f"Training complete. Total time: {fmt(total_time)}")
print(f"Best Val Macro-F1: {db_best_macro:.4f}")
print(f"{'='*65}")


# CELL 7 — FINAL EVALUATION
eval_start = time.time()

print("\nLoading best model for final test evaluation...")
db_model.load_state_dict(torch.load(
    f"{SAVE}deberta_v3_optimal_best.pt",
    map_location=device,
    weights_only=True
))
# Fix C: ensure FP32 after reload
db_model = db_model.float()

# Re-tune thresholds on val set using loaded best model
db_model.eval()
all_logits_val, all_labels_val = [], []
with torch.no_grad():
    for b in val_loader:
        db_vlogits = db_model(b["input_ids"], b["attention_mask"])
        all_logits_val.append(db_vlogits.cpu().numpy())
        all_labels_val.append(b["labels"].cpu().numpy())

all_probs_val  = 1 / (1 + np.exp(-np.vstack(all_logits_val)))
all_labels_val = np.vstack(all_labels_val)

final_thresholds = tune_threshold_per_genre(
    all_probs_val, all_labels_val, NUM_LABELS, TARGET_GENRES
)

evaluate(db_model, test_loader, final_thresholds, "DeBERTa-v3-Optimal", device)

print(f"\nEvaluation done in {fmt(time.time() - eval_start)}")
print(f"Total session time: {fmt(time.time() - train_start)}")

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.5 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
GPU:     Tesla T4
VRAM:    15.6 GB
Train: 36104, Val: 7774, Test: 7740
Labels: 12 | Device: cuda
Using: cuda


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


--- Loading data directly into GPU VRAM ---
Tokenizing 36104 texts → VRAM...
Tokenizing 7774 texts → VRAM...
Tokenizing 7740 texts → VRAM...
All splits loaded in 0:00:25
VRAM after data load: 0.2 GB / 15.6 GB (1.4% used)
Train batches: 1129 | Val: 243 | Test: 242


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VRAM after model load: 1.0 GB / 15.6 GB (6.1% used)
DeBERTa-v3-base optimal training started.
Total steps: 9032 | Steps/epoch: 1129
Batch size: 32 | Max len: 256 | Precision: FP32
Epochs: 8 | Patience: 5 | Scheduler: Cosine
Gradient checkpointing: ON (required for FP32 DeBERTa on T4)
No autocast, no GradScaler (DeBERTa-v3 FP32 constraint)
Threshold: per-genre sweep 0.20 → 0.85
Estimated total time: ~3-5 hours

─────────────────────────────────────────────────────────────────
EPOCH 1/8  |  Elapsed: 0:00:00
─────────────────────────────────────────────────────────────────


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Step  100/1129  |  Loss: 1.2258  |  Step: 2.48s  |  Epoch ETA: 0:41:05  |  Total ETA: 5:56:42
  Step  200/1129  |  Loss: 1.2098  |  Step: 2.47s  |  Epoch ETA: 0:37:41  |  Total ETA: 5:58:23
  Step  300/1129  |  Loss: 1.1797  |  Step: 2.47s  |  Epoch ETA: 0:33:49  |  Total ETA: 5:56:21
  Step  400/1129  |  Loss: 1.1526  |  Step: 2.47s  |  Epoch ETA: 0:29:50  |  Total ETA: 5:53:17
  Step  500/1129  |  Loss: 1.1257  |  Step: 2.47s  |  Epoch ETA: 0:25:47  |  Total ETA: 5:49:44
  Step  600/1129  |  Loss: 1.0943  |  Step: 2.47s  |  Epoch ETA: 0:21:42  |  Total ETA: 5:45:59
  Step  700/1129  |  Loss: 1.0655  |  Step: 2.47s  |  Epoch ETA: 0:17:36  |  Total ETA: 5:42:08
  Step  800/1129  |  Loss: 1.0384  |  Step: 2.47s  |  Epoch ETA: 0:13:31  |  Total ETA: 5:38:12
  Step  900/1129  |  Loss: 1.0169  |  Step: 2.46s  |  Epoch ETA: 0:09:24  |  Total ETA: 5:34:15
  Step 1000/1129  |  Loss: 1.0016  |  Step: 2.46s  |  Epoch ETA: 0:05:18  |  Total ETA: 5:30:14
  Step 1100/1129  |  Loss: 0.9816  |  St